<a href="https://colab.research.google.com/github/prashantdadhich/MSAI_631/blob/main/recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================
# MOVIE RECOMMENDATION SYSTEM (HYBRID MODEL)
# Item-Based Collaborative Filtering + Content-Based Fallback
# ============================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# ============================================
# LOAD DATA
# ============================================

movies = pd.read_csv("/content/movies.csv")
ratings = pd.read_csv("/content/ratings.csv")

# Merge datasets
data = pd.merge(ratings, movies, on="movieId")

# ============================================
# CREATE USER-ITEM MATRIX
# ============================================

user_item_matrix = data.pivot_table(
    index="userId",
    columns="title",
    values="rating"
)

user_item_matrix_filled = user_item_matrix.fillna(0)

# ============================================
# ITEM-BASED COLLABORATIVE FILTERING
# ============================================

item_similarity = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix_filled.columns,
    columns=user_item_matrix_filled.columns
)

def collaborative_recommend(movie_title, num_recs=5):
    if movie_title not in item_similarity_df.index:
        return None
    similar_scores = item_similarity_df[movie_title].sort_values(ascending=False)
    similar_scores = similar_scores.drop(movie_title)
    return similar_scores.head(num_recs).index.tolist()

# ============================================
# CONTENT-BASED FALLBACK (TF-IDF)
# ============================================

tfidf = TfidfVectorizer(stop_words="english")
movies["genres"] = movies["genres"].fillna("")
tfidf_matrix = tfidf.fit_transform(movies["genres"])

content_similarity = cosine_similarity(tfidf_matrix)

content_sim_df = pd.DataFrame(
    content_similarity,
    index=movies["title"],
    columns=movies["title"]
)

def content_recommend(movie_title, num_recs=5):
    if movie_title not in content_sim_df.index:
        return ["Movie not found in dataset."]
    similar_scores = content_sim_df[movie_title].sort_values(ascending=False)
    similar_scores = similar_scores.drop(movie_title)
    return similar_scores.head(num_recs).index.tolist()

# ============================================
# HYBRID RECOMMENDER
# ============================================

def hybrid_recommend(movie_title, num_recs=5):
    collab = collaborative_recommend(movie_title, num_recs)
    if collab is not None:
        return collab
    return content_recommend(movie_title, num_recs)

# ============================================
# DEMONSTRATION (NO GUI)
# ============================================

# Example usage of the hybrid recommender
print("Demonstrating the Movie Recommendation System (without GUI):\n")

sample_movie = "Toy Story (1995)"
recommendations = hybrid_recommend(sample_movie)

if recommendations:
    print(f"Recommendations for '{sample_movie}':")
    for i, movie in enumerate(recommendations):
        print(f"{i+1}. {movie}")
else:
    print(f"No recommendations found for '{sample_movie}'.")


Demonstrating the Movie Recommendation System (without GUI):

Recommendations for 'Toy Story (1995)':
1. Heat (1995)
2. GoldenEye (1995)
3. Waiting to Exhale (1995)
4. Jumanji (1995)
5. Sabrina (1995)
